# 05 — Agentes e Orquestração PDCA

Cheguei a uma conclusão importante ao longo do projeto: modelos entregam
números, mas quem toma decisão precisa de **decisões**, não de números soltos.

Este notebook demonstra a camada que construí para transformar as previsões
dos notebooks 03 e 04 em um plano de ação real.

In [ ]:
import sys, pathlib, warnings

# Descobre a pasta src/orion subindo a partir do diretório atual do kernel.
# Evita o erro "No module named 'orion'": o VS Code às vezes abre o notebook
# com o cwd na raiz do projeto, às vezes em notebooks/ — um caminho relativo
# fixo como '../src' só funciona no segundo caso.
_cwd = pathlib.Path.cwd()
for _base in [_cwd, *_cwd.parents]:
    _src = _base / 'src'
    if (_src / 'orion').is_dir():
        sys.path.insert(0, str(_src))
        break
else:
    raise FileNotFoundError(
        f"Não encontrei a pasta src/orion a partir de {_cwd}. "
        "Rode o notebook com o kernel na raiz do projeto (orion-aiops/) ou em notebooks/."
    )

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

In [ ]:
from orion.orchestrator import OrionOrchestrator, carregar_gold
from orion.agents.incidentes import AgenteIncidentes
from orion.agents.risco_ola import AgenteRiscoOLA
from orion.agents.capacidade import AgenteCapacidade
from orion.agents.performance import AgentePerformance
from orion.agents.aprendizado import AgenteAprendizado

gold = carregar_gold()
print('Tabelas Gold carregadas:')
for nome, df in gold.items():
    print(f'  {nome:32s} {df.shape}')

## 1. Anatomia de um agente

Antes de entrar no código, vale explicar o que é PDCA, já que está no título
do notebook: é uma sigla de gestão de qualidade — Plan (planejar), Do
(fazer), Check (checar), Act (agir). É um ciclo, não uma lista de passo
único: depois de agir, volta pro planejamento com o que foi aprendido. No
projeto, cada uma dessas quatro letras vira uma parte do sistema, que eu
mostro ao longo deste notebook.

Para não construir cinco agentes com cinco estruturas diferentes, defini
um contrato único que todo agente ORION precisa seguir:

```
observar()  ->  lê sua fatia da camada Gold          (PLAN)
decidir()   ->  aplica regra/modelo, gera Sinais      (DO)
executar()  ->  orquestra os dois e registra timestamp
```

Cada agente sempre devolve `list[Sinal]` — uma dataclass com severidade,
mensagem, ação recomendada e evidência em JSON. Foi esse contrato comum
que me permitiu consolidar cinco agentes heterogêneos numa fila única,
sem precisar de um tratamento especial para cada um.

In [ ]:
agente = AgenteIncidentes(gold)
estado = agente.observar()
estado['previsao'][['data_alvo', 'prioridade', 'horizonte',
                    'volume_previsto', 'ref_media', 'desvio_pct', 'z']]

In [ ]:
for s in agente.decidir(estado):
    print(f'[{s.severidade.upper():8s}] {s.titulo}')
    print(f'  {s.mensagem}')
    print(f'  Ação: {s.acao_recomendada}\n')

## 2. Agente de Risco de OLA

Este agente usa o modelo do notebook 04 para montar a fila priorizada do
dia e projetar o atingimento anual da meta — ligando o score de risco
diretamente à métrica de negócio que encontrei no notebook 01.

In [ ]:
ag_ola = AgenteRiscoOLA(gold)
est = ag_ola.observar()
print(f"Fila do dia {est['ultimo_dia']:%d/%m/%Y}: {len(est['fila'])} incidentes priorizados\n")
est['fila'][['incidente_id', 'prioridade', 'grupo_designado',
             'item_configuracao', 'score_risco', 'faixa_risco']].head(10)

In [ ]:
for s in ag_ola.decidir(est):
    print(f'[{s.severidade.upper():8s}] {s.titulo}')
    print(f'  {s.mensagem}\n')

## 3. Agente de Capacidade

Aqui tomei uma decisão de design: comparar a carga recente de cada equipe
com a **própria linha de base**, e não com outras equipes. Comparar Team14
com Team06 em valor absoluto não diria nada, porque operam em escalas
completamente diferentes.

In [ ]:
ag_cap = AgenteCapacidade(gold)
est = ag_cap.observar()
grupos = est['grupos'].sort_values('variacao', ascending=False)
grupos[['grupo_designado', 'carga_dia_base', 'carga_dia_recente',
        'variacao', 'taxa_violacao']].head(10).round(3)

In [ ]:
g = grupos.dropna(subset=['variacao']).head(12)
fig, ax = plt.subplots(figsize=(11, 5))
cores = ['#EF4444' if v >= 0.25 else '#F59E0B' if v >= 0.10 else '#38BDF8' for v in g['variacao']]
ax.barh(g['grupo_designado'], g['variacao'] * 100, color=cores)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('variação da carga vs. linha de base (%)')
ax.set_title('Top 12 equipes por variação de carga — vermelho ≥25%, laranja ≥10%')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

In [ ]:
for s in ag_cap.decidir(est)[:5]:
    print(f'[{s.severidade.upper():8s}] {s.titulo}')
    print(f'  {s.mensagem}\n')

## 4. Agente de Performance — a voz do executivo

Este agente incorpora diretamente o achado mais importante que tive no
notebook 01: ele não reporta taxa de violação, reporta **distância até o
próximo degrau de meta** — porque foi isso que aprendi que realmente
importa para o contrato.

In [ ]:
ag_perf = AgentePerformance(gold)
for s in ag_perf.executar():
    print(f'[{s.severidade.upper():8s}] {s.titulo}')
    print(f'  {s.mensagem}')
    print(f'  Ação: {s.acao_recomendada}')
    print(f'  Evidência: {s.evidencia}\n')

## 5. Agente de Aprendizado — o loop que fecha o PDCA

Na ideação do projeto, prometi entregar um sistema que aprende com o tempo.
Percebi que um pipeline que treina uma vez e nunca mais se olha no espelho
não aprende — só envelhece.

Este agente compara o que foi previsto contra o que de fato aconteceu,
detecta viés sistemático e mudança de regime, e dispara um alerta de
retreino quando necessário.

In [ ]:
ag_apr = AgenteAprendizado(gold)
for s in ag_apr.executar():
    icone = {'info': '✓', 'atencao': '!', 'alto': '!!', 'critico': '!!!'}[s.severidade]
    print(f'{icone:4s} {s.titulo}')
    print(f'     {s.mensagem}\n')

Fiquei satisfeito ao ver que o próprio agente conseguiu redescobrir sozinho
a limitação que eu já tinha identificado manualmente no notebook 03: **viés
positivo nas previsões de dezembro**, porque o modelo não conhece feriados.
É uma prova de que o sistema de monitoramento funciona — a limitação é
real, identificada automaticamente, e não escondida.

## 6. Ciclo completo do orquestrador

Com os cinco agentes funcionando individualmente, chega o momento de rodar
todos juntos e ver o resultado consolidado.

In [ ]:
orq = OrionOrchestrator(gold)
sinais = orq.ciclo()

print(f'{len(sinais)} sinais gerados por {sinais["agente"].nunique()} agentes')
print(f'{int((sinais["peso"] >= 2).sum())} exigem ação\n')
sinais[['agente', 'severidade', 'tipo', 'titulo', 'entidade', 'horizonte']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

ordem = ['info', 'atencao', 'alto', 'critico']
cores = {'info': '#94A3B8', 'atencao': '#F59E0B', 'alto': '#EF4444', 'critico': '#7F1D1D'}
sev = sinais['severidade'].value_counts().reindex(ordem).fillna(0)
axes[0].bar(sev.index, sev.values, color=[cores[s] for s in sev.index])
axes[0].set_title('Sinais por severidade')

por_ag = sinais['agente'].value_counts()
axes[1].barh(por_ag.index, por_ag.values, color='#38BDF8')
axes[1].set_title('Sinais por agente'); axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

## 7. A saída que vai para a operação

Por fim, gero o plano de ação em texto — é essa mensagem que, na minha
proposta, chegaria ao Teams ou ao e-mail do gestor todos os dias.

In [ ]:
print(orq.plano_de_acao(top=6))

In [ ]:
orq.salvar()
print('Sinais gravados em data/gold/sinais_agentes.parquet — prontos para o Power BI.')

## 8. Falha isolada

Um cuidado de engenharia que considerei importante: um agente que quebra
não pode derrubar os outros quatro. Por isso o orquestrador captura a
exceção agente por agente e registra tudo em `orq.erros`. Testo esse
comportamento simulando uma falha proposital abaixo.

In [ ]:
class AgenteQuebrado(AgenteIncidentes):
    nome = 'AgenteQuebrado'
    def observar(self):
        raise ValueError('simulação de falha: tabela Gold indisponível')

orq_teste = OrionOrchestrator(gold)
orq_teste.agentes.append(AgenteQuebrado(gold))
resultado = orq_teste.ciclo()

print(f'Sinais gerados mesmo com um agente falhando: {len(resultado)}')
print(f'Erros capturados: {len(orq_teste.erros)}')
for e in orq_teste.erros:
    print(f"  {e['agente']}: {e['erro']}")

---

## Conclusão do ciclo de notebooks

Olhando para trás, este é o resumo do que fiz em cada etapa:

| Notebook | Entrega |
|---|---|
| 01 | 6 achados críticos, incluindo a quebra de regime e a divergência de OLA |
| 02 | Duas bases de features, com vazamento de alvo demonstrado e eliminado |
| 03 | Modelo D+1/D+7 batendo os baselines nas 4 combinações |
| 04 | Risco de OLA com lift 19x e ponto de operação negociável |
| 05 | 5 agentes gerando plano de ação priorizado |

A execução completa em produção, fora do Jupyter, roda com:
`python run_pipeline.py`